## Lab 3: Control Flow for Data Cleaning (R)
This notebook implements data cleaning, error handling, and benchmarking for the Heart Disease dataset.

In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
# 1. Setup & Data Loading
if (!require("dplyr")) install.packages("dplyr", repos = "http://cran.us.r-project.org")
library(dplyr)

# Load dataset (Cleveland subset)
url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns <- c("age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target")
heart_df <- read.csv(url, header = FALSE, col.names = columns, na.strings = "?")

# Focus on trestbps and inject messy data
set.seed(42)
n <- nrow(heart_df)
# Inject negative values
heart_df$trestbps[sample(1:n, 5)] <- -120
# Inject missing values
heart_df$trestbps[sample(1:n, 5)] <- NA
# Inject extreme outliers
heart_df$trestbps[sample(1:n, 5)] <- 350

cat("Initial trestbps summary (with messy data):\n")
print(summary(heart_df$trestbps))

Initial trestbps summary (with messy data):
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
   -120     120     130     132     140     350       4 


Loading required package: dplyr

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [ ]:
%%R
# 2. Task 1: Custom BP-Cleaning Function
clean_bp <- function(x) {
  if (is.na(x)) {
    return(NA)
  } else if (x < 0) {
    return(NA)
  } else if (x > 250) {
    return(250)
  } else {
    return(x)
  }
}

In [ ]:
%%R
# 3. Task 2: Error Handling with tryCatch()
safe_stats <- function(chol, bp) {
  tryCatch({
    if (any(is.na(c(chol, bp)))) stop("NA values encountered")
    if (bp == 0) stop("Zero denominator")
    ratio <- chol / bp
    return(ratio)
  }, error = function(e) {
    warning(paste("Skipped calculation:", e$message))
    return(NA)
  })
}

# Example usage
cat("Testing safe_stats with zero BP:\n")
print(safe_stats(200, 0))

Testing safe_stats with zero BP:
[1] NA


In addition: Warning message:
In value[[3L]](cond) : Skipped calculation: Zero denominator


In [ ]:
%%R
# 4. Task 3: Loop-Based vs. Vectorized Comparison
# Loop-Based
loop_cleaning <- function(df) {
  cleaned <- df$trestbps
  for (i in 1:length(cleaned)) {
    cleaned[i] <- clean_bp(cleaned[i])
  }
  return(cleaned)
}

# Vectorized
vectorized_cleaning <- function(df) {
  cleaned <- ifelse(is.na(df$trestbps), NA,
                    ifelse(df$trestbps < 0, NA,
                           ifelse(df$trestbps > 250, 250, df$trestbps)))
  return(cleaned)
}

# Benchmarking
cat("Benchmarking Loop Method:\n")
print(system.time(replicate(100, loop_cleaning(heart_df))))

cat("\nBenchmarking Vectorized Method:\n")
print(system.time(replicate(100, vectorized_cleaning(heart_df))))

Benchmarking Loop Method:
   user  system elapsed 
  0.033   0.001   0.034 

Benchmarking Vectorized Method:
   user  system elapsed 
  0.012   0.000   0.012 


In [ ]:
%%R
# 5. Task 4: Validation & Summary Statistics
heart_df$trestbps_cleaned <- vectorized_cleaning(heart_df)

# Validation checks
na_count <- sum(is.na(heart_df$trestbps_cleaned))
neg_count <- sum(heart_df$trestbps_cleaned < 0, na.rm = TRUE)
outlier_count <- sum(heart_df$trestbps_cleaned > 250, na.rm = TRUE)

cat("Validation Statistics:\n")
cat("Missing values:", na_count, "\n")
cat("Negative values:", neg_count, "\n")
cat("Values > 250:", outlier_count, "\n")

# Assertions
stopifnot(neg_count == 0)
stopifnot(outlier_count == 0)

cat("\nSummary of Cleaned trestbps:\n")
print(summary(heart_df$trestbps_cleaned))

# Export Deliverables
write.csv(heart_df, "cleaned_heart_data.csv", row.names = FALSE)
cat("\nData exported to cleaned_heart_data.csv\n")

Validation Statistics:
Missing values: 8 
Negative values: 0 
Values > 250: 0 

Summary of Cleaned trestbps:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
   94.0   120.0   130.0   133.8   140.0   250.0       8 

Data exported to cleaned_heart_data.csv
